# Ko-miracl RAG — 생성(Generation) 파이프라인

앞 노트북(`ko_miracl_rag.ipynb`)의 **임베딩 · ChromaDB · retriever** 코드를 재사용하고,
그 위에 **생성형 RAG** 파이프라인을 얹습니다. (LangChain 미사용)

```
사용자의 추상적 질문
   └─▶ ① LLM 질의 재작성 (rewrite)
        └─▶ ② retriever 검색 (ChromaDB, bge-m3)
             └─▶ ③ 프롬프트 생성 (검색 문서 주입)
                  └─▶ ④ LLM 응답 생성
```

- **임베딩(검색)**: `BAAI/bge-m3`
- **LLM(생성)**: `Qwen/Qwen2.5-7B-Instruct` (한국어 지원, Colab T4에서 4-bit 로드)

## 0. 설치 · 설정 (재사용 파트)

In [1]:
!pip -q install -U datasets chromadb sentence-transformers transformers accelerate bitsandbytes

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 5.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 555.1/555.1 kB 22.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.3/23.3 MB 100.4 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.6/11.6 MB 138.6 MB/s eta 0:00:0000:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 43.7 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 28.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.6/4.6 MB 123.3 MB/s eta 0:00:0000:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.7/18.7 MB 122.4 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.5/72.5 kB 8.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.2/137.2 kB 13.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.0/60.0 kB 6.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 204.6/204.6 kB 22.7 MB/s eta 0:00:0

In [2]:
import random, numpy as np, pandas as pd, torch
random.seed(42); np.random.seed(42)
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", DEVICE)

# ===== 검색 임베딩 모델 (단일: bge-m3) =====
MODEL_CFG = {"model_name": "BAAI/bge-m3", "query_prefix": "", "passage_prefix": ""}

# ===== 생성 LLM (한국어 지원) =====
LLM_NAME = "Qwen/Qwen2.5-7B-Instruct"   # 메모리 부족 시 'Qwen/Qwen2.5-3B-Instruct' 로 교체

# ===== 하이퍼파라미터 =====
TOP_K = 5
N_EVAL_QUERIES = 30      # 인덱싱할 코퍼스 규모를 줄이려고 작게 설정
NEG_POOL_SIZE  = 3000
BATCH_SIZE = 128
MAX_SEQ_LEN = 512

Device: cuda


In [3]:
from datasets import load_dataset

# queries: 질문 목록 / default(dev): 관련도 라벨 → 데모용 인덱스 구성에 사용
q_dd = load_dataset("taeminlee/Ko-miracl", "queries")
queries = {r["_id"]: r["text"] for r in q_dd[list(q_dd.keys())[0]]}

default_dd = load_dataset("taeminlee/Ko-miracl", "default")
qrels_df = pd.DataFrame(default_dd["dev"])
print("queries:", len(queries), "| qrels(dev):", len(qrels_df))

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:138: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
  warnings.warn(f"\nError while fetching `HF_TOKEN` secret value from your vault: '{str(e)}'.")


README.md:   0%|          | 0.00/1.66k [00:00<?, ?B/s]

queries.jsonl:   0%|          | 0.00/219k [00:00<?, ?B/s]

Generating queries split:   0%|          | 0/2761 [00:00<?, ? examples/s]

train.jsonl:   0%|          | 0.00/641k [00:00<?, ?B/s]

dev.jsonl:   0%|          | 0.00/153k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/12767 [00:00<?, ? examples/s]

Generating dev split:   0%|          | 0/3057 [00:00<?, ? examples/s]

queries: 2761 | qrels(dev): 3057


### 코퍼스 부분집합 인덱싱 (재사용)
전체 149만 청크 대신 **정답 문서 + 네거티브 풀**만 스트리밍으로 수집합니다.
스트리밍이라 이 셀이 가장 오래 걸립니다 (수 분). 빠르게 보려면 `N_EVAL_QUERIES`를 줄이세요.

In [4]:
qrels_pos = qrels_df[qrels_df["score"] > 0]
eval_qids = qrels_pos["query-id"].unique().tolist()
random.shuffle(eval_qids); eval_qids = eval_qids[:N_EVAL_QUERIES]

gold = {}
for _, row in qrels_df[qrels_df["query-id"].isin(eval_qids)].iterrows():
    gold.setdefault(row["query-id"], {})[row["corpus-id"]] = float(row["score"])
needed_cids = {c for rels in gold.values() for c, s in rels.items() if s > 0}
print("데모 쿼리:", len(eval_qids), "| 정답 문서:", len(needed_cids))

데모 쿼리: 30 | 정답 문서: 100


In [5]:
corpus_dd = load_dataset("taeminlee/Ko-miracl", "corpus", streaming=True)
corpus_stream = corpus_dd[list(corpus_dd.keys())[0]]

docs, found, neg = {}, set(), 0
for row in corpus_stream:
    cid = row["_id"]
    if cid in needed_cids and cid not in docs:
        docs[cid] = {"title": row["title"], "text": row["text"]}; found.add(cid)
    elif neg < NEG_POOL_SIZE:
        docs[cid] = {"title": row["title"], "text": row["text"]}; neg += 1
    if len(found) >= len(needed_cids) and neg >= NEG_POOL_SIZE:
        break
print("수집 청크:", len(docs), "| 정답 확보:", len(found), "/", len(needed_cids))

수집 청크: 3100 | 정답 확보: 100 / 100


### 임베딩 · ChromaDB · retriever (재사용, 단일 모델)

In [6]:
from sentence_transformers import SentenceTransformer
import chromadb

def load_model(cfg):
    m = SentenceTransformer(cfg["model_name"], device=DEVICE); m.max_seq_length = MAX_SEQ_LEN; return m

def embed(model, texts, prefix, batch_size=BATCH_SIZE, show_progress=True):
    texts = [prefix + t for t in texts]
    return model.encode(texts, batch_size=batch_size, normalize_embeddings=True,
                        convert_to_numpy=True, show_progress_bar=show_progress)

chroma_client = chromadb.Client()

def build_collection(name, model, cfg, docs):
    try: chroma_client.delete_collection(name)
    except Exception: pass
    col = chroma_client.create_collection(name=name, metadata={"hnsw:space": "cosine"})
    cids = list(docs.keys())
    texts = [docs[c]["text"] for c in cids]; titles = [docs[c]["title"] for c in cids]
    embs = embed(model, texts, cfg["passage_prefix"])
    for i in range(0, len(cids), BATCH_SIZE):
        col.add(ids=cids[i:i+BATCH_SIZE], embeddings=embs[i:i+BATCH_SIZE].tolist(),
                documents=texts[i:i+BATCH_SIZE],
                metadatas=[{"title": t} for t in titles[i:i+BATCH_SIZE]])
    print(f"[{name}] 적재 완료: {col.count()} chunks"); return col

def retrieve(col, model, cfg, query_text, k=TOP_K):
    q_emb = embed(model, [query_text], cfg["query_prefix"], batch_size=1, show_progress=False)
    res = col.query(query_embeddings=q_emb.tolist(), n_results=k)
    out = []
    for cid, doc, dist, meta in zip(res["ids"][0], res["documents"][0],
                                    res["distances"][0], res["metadatas"][0]):
        out.append({"corpus_id": cid, "score": 1 - dist,          # score = 1 - distance
                    "title": meta.get("title", ""), "text": doc})  # 생성용이라 전체 텍스트 유지
    return out

In [7]:
model = load_model(MODEL_CFG)
collection = build_collection("ko-miracl-bge-m3", model, MODEL_CFG, docs)

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/123 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/15.8k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/54.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/687 [00:00<?, ?B/s]

...

tokenizer.json: downloading bytes:           |  0.00B            

special_tokens_map.json:   0%|          | 0.00/964 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/191 [00:00<?, ?B/s]

Batches:   0%|          | 0/25 [00:00<?, ?it/s]

[ko-miracl-bge-m3] 적재 완료: 3100 chunks


## 1. 생성 LLM 로드 (한국어)
`Qwen2.5-7B-Instruct`를 4-bit로 로드합니다.

In [8]:
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

bnb = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4",
                         bnb_4bit_compute_dtype=torch.float16, bnb_4bit_use_double_quant=True)
tok = AutoTokenizer.from_pretrained(LLM_NAME)
llm = AutoModelForCausalLM.from_pretrained(LLM_NAME, quantization_config=bnb, device_map="auto")
llm.eval()
print("LLM loaded:", LLM_NAME)

config.json:   0%|          | 0.00/663 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

...

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:213: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


generation_config.json:   0%|          | 0.00/243 [00:00<?, ?B/s]

LLM loaded: Qwen/Qwen2.5-7B-Instruct


In [9]:
@torch.no_grad()
def chat(system, user, max_new_tokens=512, temperature=0.3):
    msgs = [{"role": "system", "content": system}, {"role": "user", "content": user}]
    # 최신 transformers는 return_dict=True로 BatchEncoding을 받아 **enc로 넘기는 게 안전
    enc = tok.apply_chat_template(msgs, add_generation_prompt=True,
                                  return_tensors="pt", return_dict=True).to(llm.device)
    out = llm.generate(**enc, max_new_tokens=max_new_tokens,
                       do_sample=temperature > 0, temperature=max(temperature, 1e-5),
                       top_p=0.9, pad_token_id=tok.eos_token_id)
    gen = out[0][enc["input_ids"].shape[1]:]
    return tok.decode(gen, skip_special_tokens=True).strip()

## 2. ① 질의 재작성 (Query Rewriting)
사용자의 추상적/모호한 질문을 검색에 적합한 핵심 질의로 재작성합니다.

In [10]:
def rewrite_query(question):
    system = ("당신은 검색 질의 재작성 전문가입니다. 사용자의 모호하거나 추상적인 질문을 "
              "검색엔진에 적합하도록 핵심 개체·키워드 중심의 명확한 한국어 검색 질의로 바꾸세요. "
              "설명 없이 재작성된 질의 한 줄만 출력하세요.")
    user = f"사용자 질문: {question}\n재작성된 검색 질의:"
    return chat(system, user, max_new_tokens=64, temperature=0.0)

## 3. ③ 프롬프트 생성
검색된 문서를 근거로 주입하는 RAG 프롬프트를 만듭니다.

In [11]:
def build_prompt(question, contexts, snippet_len=800):
    ctx = "\n\n".join(
        f"[문서 {i+1}] (id={c['corpus_id']}, title={c['title']})\n{c['text'][:snippet_len]}"
        for i, c in enumerate(contexts))
    system = ("당신은 주어진 참고 문서를 근거로 한국어로 답하는 assistant입니다. "
              "문서에 없는 내용은 지어내지 말고, 근거가 없으면 '제공된 문서에서 찾을 수 없습니다'라고 답하세요. "
              "답변에 사용한 문서 번호를 [문서 n] 형태로 인용하세요.")
    user = f"# 참고 문서\n{ctx}\n\n# 질문\n{question}\n\n# 답변"
    return system, user

## 4. 전체 파이프라인
사용자 질문 → ① 재작성 → ② 검색 → ③ 프롬프트 → ④ 응답

In [12]:
def rag_answer(question, k=TOP_K, verbose=True):
    # ① 질의 재작성
    rq = rewrite_query(question)
    # ② retriever 검색
    ctxs = retrieve(collection, model, MODEL_CFG, rq, k=k)
    # ③ 프롬프트 생성
    system, user = build_prompt(question, ctxs)
    # ④ LLM 응답 생성
    answer = chat(system, user, max_new_tokens=512, temperature=0.2)

    if verbose:
        print("① 원 질문     :", question)
        print("① 재작성 질의 :", rq)
        print("② 검색 결과 (score = 1 - distance):")
        for i, c in enumerate(ctxs, 1):
            print(f"   [{i}] score={c['score']:.4f} id={c['corpus_id']} | {c['title']}")
        print("\n④ 생성 응답 :\n" + answer)
    return {"question": question, "rewritten": rq, "contexts": ctxs, "answer": answer}

## 5. 실행 예시

In [13]:
# 예시 A: Ko-miracl dev의 실제 질문 (정답 문서가 인덱스에 포함되어 있음)
_ = rag_answer(queries[eval_qids[0]])

[transformers] The following generation flags are not valid and may be ignored: ['temperature', 'top_p']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:468: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


① 원 질문     : 구지가는 고대 시가의 하나인가요?
① 재작성 질의 : 구지가가 고대 시인가요?
② 검색 결과 (score = 1 - distance):
   [1] score=0.6918 id=264838#12 | 한국의 상고시대 시가
   [2] score=0.4292 id=986864#0 | 양해 (후한)
   [3] score=0.4098 id=987124#0 | 요시카와 코지로
   [4] score=0.3878 id=989604#0 | 트라시마코스
   [5] score=0.3871 id=988540#71 | 왕자 조

④ 생성 응답 :
네, 구지가는 고대 시가의 하나입니다. [문서 1]에 따르면, 〈구지가(龜旨歌)〉는 서기 40년경에 이루어진 고대 시가 중 하나입니다.


In [18]:
_ = rag_answer(queries[eval_qids[12]])

① 원 질문     : 한국어는 언제 만들어졌나요?
① 재작성 질의 : 한국어가 언제 시작되었나요?
② 검색 결과 (score = 1 - distance):
   [1] score=0.6093 id=487516#0 | 한글
   [2] score=0.4787 id=1852#9 | 유럽
   [3] score=0.4708 id=261019#0 | 불교의 역사
   [4] score=0.4509 id=987704#3 | 학생신문
   [5] score=0.4459 id=3966#1 | 한국의 성씨와 이름

④ 생성 응답 :
한국어를 표기하기 위해 만들어진 문자는 1446년에 창제되었습니다. [문서 1]에 따르면, 조선 제 4대 임금 세종이 훈민정음(训民正音)이라는 이름으로 문자를 창제하고 1446년에 반포하였습니다.


In [14]:
# 예시 B: 사용자가 던지는 추상적/모호한 질문
_ = rag_answer("그 유명한 상대성 이론 만든 사람 어떤 인물이야?")

① 원 질문     : 그 유명한 상대성 이론 만든 사람 어떤 인물이야?
① 재작성 질의 : 상대성 이론을 만든 사람 누구인가요?
② 검색 결과 (score = 1 - distance):
   [1] score=0.4842 id=3533#0 | 르네 데카르트
   [2] score=0.4504 id=780040#17 | 라이트 판타스틱
   [3] score=0.4304 id=987221#0 | 류드비크 파데예프
   [4] score=0.4189 id=987336#0 | 에드워드 손다이크
   [5] score=0.4139 id=985079#3 | 마이어-피토리스 열

④ 생성 응답 :
제공된 문서에서 찾을 수 없습니다. 상대성 이론은 알베르트 아인슈타인에 의해 만들어졌으나, 제공된 문서들 중에서는 아인슈타인에 대한 정보가 없습니다.


In [15]:
_ = rag_answer("물 몇도에 끓음?")

/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:468: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


① 원 질문     : 물 몇도에 끓음?
① 재작성 질의 : 물이 끓는 온도는 몇 도인가요?
② 검색 결과 (score = 1 - distance):
   [1] score=0.5234 id=986668#2 | 2003년 유럽 폭염
   [2] score=0.5144 id=985918#0 | 로스팅
   [3] score=0.4879 id=987625#13 | 주위
   [4] score=0.4788 id=47536#1 | 우라늄
   [5] score=0.4788 id=986668#1 | 2003년 유럽 폭염

④ 생성 응답 :
제공된 문서에서 찾을 수 없습니다. 끓임溫度的相关信息没有在给定的文档中提及。


In [16]:
_ = rag_answer("키보드 주문제작 하고 싶어")

① 원 질문     : 키보드 주문제작 하고 싶어
① 재작성 질의 : 키보드 제작 서비스 찾기
② 검색 결과 (score = 1 - distance):
   [1] score=0.6641 id=985068#3 | 커스텀 키보드
   [2] score=0.6520 id=985068#4 | 커스텀 키보드
   [3] score=0.6363 id=985068#2 | 커스텀 키보드
   [4] score=0.6326 id=985068#0 | 커스텀 키보드
   [5] score=0.6277 id=985068#1 | 커스텀 키보드

④ 생성 응답 :
키보드 주문제작을 원하신다면 두 가지 방법이 있습니다. 하나는 직접 설계와 제작을 하며, 다른 하나는 공동으로 제작하는 방법입니다. 직접 설계 및 제작하는 방법은 전문 지식이 필요하고 비용이 많이 들기 때문에, 대부분의 사용자들은 공동 제작을 선택하는 경향이 있습니다. 공동 제작을 통해 전문적인 지식 없이 비교적 저렴하게 커스텀 키보드를 제작할 수 있습니다. [문서 1], [문서 2] 참조.


## 정리
- **재사용**: 앞 노트북의 `embed` / `build_collection` / `retrieve`(ChromaDB, `score=1-distance`)를 그대로 사용.
- **추가**: `rewrite_query`(①) → `retrieve`(②) → `build_prompt`(③) → `chat`(④) 로 이어지는 생성 RAG.
- **LLM**: 한국어 지원 `Qwen2.5-7B-Instruct` (4-bit). 더 가볍게: `Qwen2.5-3B-Instruct`, 또는 `LGAI-EXAONE/EXAONE-3.5-7.8B-Instruct` 등으로 교체 가능.

### 더 해볼 것
- 재작성을 **다중 질의(multi-query)** 로 확장해 검색 recall 향상
- 리랭커(`BAAI/bge-reranker-v2-m3`)로 검색 결과 재정렬 후 상위 문서만 프롬프트에 주입
- 답변의 인용 문서와 정답셋(`gold`)을 비교해 groundedness 평가